# 9.1 — Exploração de Metadados ANEEL

Análise exploratória do dataset de metadados da legislação ANEEL.
Fonte: `data/processed/metadata.parquet`

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Adiciona raiz do projeto ao path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

PROCESSED_DIR = ROOT / 'data' / 'processed'

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('ROOT:', ROOT)

In [ ]:
# Carrega metadata.parquet
metadata_path = PROCESSED_DIR / 'metadata.parquet'

try:
    df = pd.read_parquet(metadata_path)
    print(f'Shape: {df.shape}')
    print(f'\nDtypes:')
    print(df.dtypes)
    print(f'\nPrimeiras linhas:')
    df.head(3)
except FileNotFoundError:
    print(f'Arquivo nao encontrado: {metadata_path}')
    print('Execute src/01_consolidate_metadata.py primeiro.')
    raise

In [ ]:
# Distribuicao por tipo de documento
tipo_counts = df['tipo_sigla'].value_counts()
print('Contagem por tipo_sigla:')
print(tipo_counts)

fig, ax = plt.subplots()
tipo_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Documentos por Tipo de Ato')
ax.set_xlabel('Tipo')
ax.set_ylabel('Quantidade')
ax.tick_params(axis='x', rotation=0)
for bar in ax.patches:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 50,
        f'{int(bar.get_height()):,}',
        ha='center', va='bottom', fontsize=9
    )
plt.tight_layout()
plt.show()

In [ ]:
# Distribuicao por ano
ano_col = 'ano' if 'ano' in df.columns else 'ano_json'
ano_counts = df[ano_col].value_counts().sort_index()
print('Documentos por ano:')
print(ano_counts)

fig, ax = plt.subplots()
ano_counts.plot(kind='bar', ax=ax, color='coral', edgecolor='white')
ax.set_title('Documentos por Ano')
ax.set_xlabel('Ano')
ax.set_ylabel('Quantidade')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Cobertura de PDFs baixados
if 'pdf_baixado' in df.columns:
    total = len(df)
    baixados = df['pdf_baixado'].sum()
    pct = baixados / total * 100
    print(f'PDFs baixados: {int(baixados):,} / {total:,} ({pct:.1f}%)')

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.pie(
        [baixados, total - baixados],
        labels=['Baixado', 'Nao baixado'],
        autopct='%1.1f%%',
        colors=['steelblue', 'lightgray'],
        startangle=90
    )
    ax.set_title('Cobertura de Download de PDFs')
    plt.tight_layout()
    plt.show()
else:
    # Conta arquivos .txt em data/processed/texts/
    texts_dir = ROOT / 'data' / 'processed' / 'texts'
    if texts_dir.exists():
        n_txt = len(list(texts_dir.rglob('*.txt')))
        print(f'Arquivos .txt em texts/: {n_txt:,}')
        print(f'Total de registros: {len(df):,}')
        print(f'Cobertura estimada: {n_txt/len(df)*100:.1f}%')
    else:
        print('Coluna pdf_baixado nao encontrada e pasta texts/ nao existe.')
        print(f'Colunas disponiveis: {df.columns.tolist()}')

In [ ]:
# Histograma de comprimento de ementa
if 'ementa' in df.columns:
    df['ementa_len'] = df['ementa'].fillna('').apply(len)
    print(df['ementa_len'].describe().round(1))

    fig, ax = plt.subplots()
    ax.hist(df['ementa_len'], bins=50, color='mediumseagreen', edgecolor='white')
    ax.axvline(df['ementa_len'].median(), color='red', linestyle='--', label=f'Mediana: {df["ementa_len"].median():.0f}')
    ax.set_title('Distribuicao de Comprimento de Ementa (chars)')
    ax.set_xlabel('Comprimento (caracteres)')
    ax.set_ylabel('Frequencia')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Coluna ementa nao encontrada.')

In [ ]:
# Top 10 assuntos mais frequentes
if 'assunto' in df.columns:
    top_assuntos = df['assunto'].fillna('(sem assunto)').value_counts().head(10)
    print('Top 10 assuntos:')
    print(top_assuntos.to_string())

    fig, ax = plt.subplots(figsize=(12, 5))
    top_assuntos.sort_values().plot(kind='barh', ax=ax, color='mediumpurple', edgecolor='white')
    ax.set_title('Top 10 Assuntos')
    ax.set_xlabel('Quantidade')
    plt.tight_layout()
    plt.show()
else:
    print('Coluna assunto nao encontrada.')

In [ ]:
# Documentos revogados vs vigentes
if 'revogada' in df.columns:
    rev_counts = df['revogada'].map({True: 'Revogada', False: 'Vigente'}).value_counts()
elif 'situacao' in df.columns:
    rev_counts = df['situacao'].fillna('Desconhecido').value_counts().head(5)
else:
    rev_counts = None
    print('Colunas revogada/situacao nao encontradas.')

if rev_counts is not None:
    print('Situacao dos documentos:')
    print(rev_counts.to_string())

    fig, ax = plt.subplots(figsize=(8, 4))
    rev_counts.plot(kind='bar', ax=ax, color=['tomato', 'steelblue'], edgecolor='white')
    ax.set_title('Situacao dos Documentos (Revogacao)')
    ax.set_xlabel('')
    ax.set_ylabel('Quantidade')
    ax.tick_params(axis='x', rotation=20)
    plt.tight_layout()
    plt.show()